In [166]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pickle
import yaml
import os
import re
import math
import getpass

In [167]:
def load_datapath(filepath):
    with open(filepath, "r") as file:
        config = yaml.safe_load(file)
    return config["paths"]["progress_dir"]

In [168]:
def load_data(folder):
    # loads most previous data with given settings from folder 
    with open(os.path.join(folder, "data.pkl"), "rb") as file:
        save_data = pickle.load(file)
        points = save_data["points"]
        momenta = save_data["momenta"]
        losses = save_data["losses"]
        gradients = save_data["gradients"]
    return points, momenta, losses, gradients

In [169]:
def extract_parts(input_string):
    pattern = r"^(?P<n_closest_points>\w+)_((?P<weighting>\w+)_(?P<fit_mode>\w+)_(?P<momentum_type>\w+)_decay(?P<decay>[\d.]+)lr(?P<lr>[\d.]+)_\d{4}-\d{2}-\d{2}_\w+)$"
    # Match the pattern with the input string
    match = re.match(pattern, input_string) 
    # Extract components into a dictionary
    if match: return match.groupdict() 
    else: raise ValueError("Input string does not match the expected format")

In [170]:
def get_common_keys(dicts):
    keys = set(dicts[0].keys())
    common_keys = {key for key in keys if all(d[key]==dicts[0][key] for d in dicts)}
    # for d in dicts:
    #     for key in common_keys:
    #         del d[key]
    # return dicts
    return common_keys

In [171]:
def create_lossCurve(folders, save_path):
    colors = {"Global":"#0000FF", "1000":"#3333FF", "250":"#6666FF","100":"#9999FF"}
    
    attribute_dict_list=[]
    for folder in folders:
        parts = extract_parts(os.path.basename(folder))
        attribute_dict_list.append(parts)
    common_keys = get_common_keys(attribute_dict_list)

    plt.figure(figsize=(8,6))

    for i, folder in enumerate(folders):
        point_list, momentum_list, loss_list, gradient_list = load_data(folder)
        label = ", ".join([f"{key}: {value}" for (key, value) in attribute_dict_list[i].items() if key not in common_keys])
        plt.plot([i for i in range(len(loss_list))], loss_list, "o--", color=colors[attribute_dict_list[i]["n_closest_points"]], label=label)
        
    plt.grid(True)
    plt.ylabel("Neg. Log-Likelihood")
    plt.xlabel("Gradient Descent Steps")
    title = ", ".join([f"{key}: {value}" for (key, value) in attribute_dict_list[0].items() if key in common_keys])
    plt.title(title)
    plt.legend() 
    plt.savefig(os.path.join(save_path, "_".join([f"{key}: {value}" for (key, value) in attribute_dict_list[0].items() if key in common_keys]))+".png")
    plt.close()

In [172]:
def find_folders(path, search_strings):
    matching_folders = []
    for root, dirs, files in os.walk(path):
        for folder in dirs:
            if all(s in folder for s in search_strings):
                matching_folders.append(os.path.join(root, folder))
    return matching_folders

In [173]:
def main():
    common_attributes = ["None", "lr0.005"]
    save_path = f"/home/{getpass.getuser()}/CALETana/MLProject/ComparisonPlots/plots/"
    config_filepath = f"/home/{getpass.getuser()}/CALETana/MLProject/GradientDescent/config.yaml"
    
    data_path = load_datapath(config_filepath)
    folders = find_folders(data_path, common_attributes)
    create_lossCurve(folders, save_path)
    
main()